# 03 · Experiment worker (run in any account, as often as you like)
Each run picks up unfinished jobs, saves progress after every round, and stops cleanly before the time budget.
Several accounts can run this **at the same time**: give each a different `WORKER` name and `SHARD` number.
If Colab disconnects, just run it again - the job resumes from its last saved round
(a job claimed by a dead session is released after 3 hours).

`Runtime -> Change runtime type -> T4 GPU`, set the four values below, then `Runtime -> Run all`.

In [ ]:
WORKER = "acc1"        # a different name per Colab account
SHARD = 0              # 0, 1, 2, ... one per account
N_SHARDS = 3           # how many accounts you plan to run in parallel
HOURS = 3.5            # stop starting new jobs after this (free Colab GPU sessions are limited)
EXPERIMENTS = None     # None = all, in priority order E1 -> E3 -> E2 -> E4; or e.g. ["E1_temporal"]

In [ ]:
# ---- setup: identical in every notebook ----
FOLDER = "AMR_AL"          # name of the shared folder (or its shortcut) in My Drive

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
ROOT = f"/content/drive/MyDrive/{FOLDER}"
if not os.path.isdir(os.path.join(ROOT, "amr_al")):
    raise SystemExit(f"{ROOT}/amr_al not found.\nIn Google Drive open 'Shared with me', right-click {FOLDER} -> "
                     "Organize -> Add shortcut -> My Drive, then run this cell again.")
try:
    import xgboost
    if int(xgboost.__version__.split(".")[0]) < 2:
        raise ImportError
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"], check=True)
os.environ["AMR_AL_ROOT"] = ROOT
sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith("amr_al")]:
    del sys.modules[m]                      # always load the latest code from Drive
from amr_al import config, paths, data, models, runner, audit, analysis, driams_fetch
paths.ensure_dirs()
print("shared folder:", ROOT)
print("compute device for XGBoost:", models.device())

In [ ]:
data.stage_to_local()
runner.status(EXPERIMENTS)

In [ ]:
runner.run_worker(WORKER, shard=SHARD, n_shards=N_SHARDS, experiments=EXPERIMENTS, hours=HOURS)

In [ ]:
runner.status(EXPERIMENTS)
drive.flush_and_unmount()      # make sure every result is written to Drive before the session ends
print("safe to close")